# 🦀 Day 1: Modules — `mod`, `pub`, `use`

This week we learn how Rust organizes code! Today:
- Module system basics
- Visibility with `pub`
- The `use` keyword
- Module tree and paths

---

## 📦 What Are Modules?

Modules let you organize code into logical groups and control visibility (public vs private).

Think of modules like folders in a file system:
```
crate (root)
├── module_a
│   ├── function_1
│   └── sub_module
│       └── function_2
└── module_b
    └── function_3
```

## 🏗️ Defining Modules with `mod`

In [ ]:
// Define a module inline
mod math {
    // By default, everything is PRIVATE to the module
    fn secret_formula(x: f64) -> f64 {
        x * 1.618  // Golden ratio
    }

    // Use `pub` to make items accessible outside the module
    pub fn add(a: f64, b: f64) -> f64 {
        a + b
    }

    pub fn multiply(a: f64, b: f64) -> f64 {
        a * b
    }

    pub fn golden(x: f64) -> f64 {
        secret_formula(x)  // Can call private fn within same module
    }
}

// Call public functions with full path
println!("3 + 4 = {}", math::add(3.0, 4.0));
println!("3 × 4 = {}", math::multiply(3.0, 4.0));
println!("golden(10) = {:.3}", math::golden(10.0));

// ❌ Can't call private function from outside:
// math::secret_formula(5.0);  // ERROR: function is private

## 🔑 Visibility Rules

| Keyword | Visibility |
|---------|------------|
| (none) | Private to current module |
| `pub` | Public — accessible from anywhere |
| `pub(crate)` | Public within the crate only |
| `pub(super)` | Public to parent module |
| `pub(in path)` | Public within a specific path |

In [ ]:
mod outer {
    pub mod inner {
        pub fn public_fn() {
            println!("  I'm public!");
            private_fn();
        }

        fn private_fn() {
            println!("  I'm private to `inner`");
        }

        pub(super) fn super_fn() {
            println!("  I'm visible to `outer` (parent)");
        }
    }

    pub fn call_inner() {
        println!("From outer:");
        inner::public_fn();
        inner::super_fn();    // ✅ Works — pub(super)
    }
}

outer::call_inner();
outer::inner::public_fn();
// outer::inner::super_fn();  // ❌ Not visible from here

## 📌 The `use` Keyword

Bring items into scope so you don't need full paths.

In [ ]:
mod geometry {
    pub const PI: f64 = 3.14159265358979;

    pub fn circle_area(radius: f64) -> f64 {
        PI * radius * radius
    }

    pub fn circle_circumference(radius: f64) -> f64 {
        2.0 * PI * radius
    }

    pub mod shapes {
        pub fn rectangle_area(w: f64, h: f64) -> f64 { w * h }
        pub fn triangle_area(base: f64, height: f64) -> f64 { 0.5 * base * height }
    }
}

// Without `use` — verbose
println!("Circle area: {:.2}", geometry::circle_area(5.0));

// With `use` — clean!
use geometry::circle_area;
use geometry::shapes::rectangle_area;
println!("Circle area: {:.2}", circle_area(5.0));
println!("Rect area: {:.2}", rectangle_area(4.0, 6.0));

In [ ]:
// Use multiple items with {}
use geometry::{circle_circumference, PI};
use geometry::shapes::{rectangle_area as rect_area, triangle_area};

println!("PI = {}", PI);
println!("Circumference: {:.2}", circle_circumference(5.0));
println!("Rect: {:.2}", rect_area(3.0, 7.0));
println!("Triangle: {:.2}", triangle_area(6.0, 4.0));

In [ ]:
// Glob import with * (use sparingly!)
use geometry::shapes::*;
println!("All imported: {}", triangle_area(10.0, 5.0));

## 🏗️ Structs & Enums in Modules

In [ ]:
mod animals {
    // Pub struct — but fields are still private by default!
    #[derive(Debug)]
    pub struct Dog {
        pub name: String,      // public field
        pub breed: String,     // public field
        age: u8,               // PRIVATE field
    }

    impl Dog {
        // Constructor needed because `age` is private
        pub fn new(name: &str, breed: &str, age: u8) -> Self {
            Dog {
                name: name.into(),
                breed: breed.into(),
                age,
            }
        }

        pub fn age(&self) -> u8 {
            self.age  // Getter for private field
        }
    }

    // Pub enum — ALL variants are public automatically
    #[derive(Debug)]
    pub enum Sound {
        Bark,
        Whine,
        Growl,
    }
}

let dog = animals::Dog::new("Rex", "German Shepherd", 5);
println!("{} the {} (age {})", dog.name, dog.breed, dog.age());

// Can access public fields directly
// println!("{}", dog.age);  // ❌ age is private

// Enum variants are always public if enum is public
let sound = animals::Sound::Bark;
println!("Sound: {:?}", sound);

## 🔄 The `self` and `super` Keywords in Paths

In [ ]:
mod parent {
    pub fn parent_fn() {
        println!("Parent function");
    }

    pub mod child {
        pub fn child_fn() {
            println!("Child function");
            // super:: refers to the parent module
            super::parent_fn();
        }

        pub fn call_sibling() {
            // self:: refers to the current module (optional but explicit)
            self::child_fn();
        }
    }
}

parent::child::child_fn();
println!();
parent::child::call_sibling();

## 📚 Practical: Library Module System

In [ ]:
mod library {
    pub mod catalog {
        #[derive(Debug)]
        pub struct Book {
            pub title: String,
            pub author: String,
            pub isbn: String,
        }

        impl Book {
            pub fn new(title: &str, author: &str, isbn: &str) -> Self {
                Book {
                    title: title.into(),
                    author: author.into(),
                    isbn: isbn.into(),
                }
            }
        }
    }

    pub mod members {
        #[derive(Debug)]
        pub struct Member {
            pub name: String,
            pub id: u32,
            borrowed_books: Vec<String>,
        }

        impl Member {
            pub fn new(name: &str, id: u32) -> Self {
                Member { name: name.into(), id, borrowed_books: Vec::new() }
            }

            pub fn borrow_book(&mut self, isbn: &str) {
                self.borrowed_books.push(isbn.to_string());
            }

            pub fn books_count(&self) -> usize {
                self.borrowed_books.len()
            }
        }
    }

    // Re-export for convenience
    pub use catalog::Book;
    pub use members::Member;
}

// Thanks to re-export, we can use shorter paths
use library::{Book, Member};

let book = Book::new("The Rust Book", "Klabnik", "978-1");
let mut member = Member::new("Alice", 1);
member.borrow_book(&book.isbn);

println!("📖 {} by {}", book.title, book.author);
println!("👤 {} has {} books", member.name, member.books_count());

## 🏋️ Exercises

In [ ]:
// Exercise 1: Create a `converter` module with sub-modules:
// - `temperature`: celsius_to_fahrenheit, fahrenheit_to_celsius
// - `distance`: km_to_miles, miles_to_km
// - `weight`: kg_to_lbs, lbs_to_kg
// Re-export the most common functions from the `converter` module

// YOUR CODE HERE


In [ ]:
// Exercise 2: Create a `game` module with:
// - A `Player` struct (public name, private health/score)
// - An `Action` enum (Attack, Heal, Shield)
// - Methods to modify health based on actions

// YOUR CODE HERE


## 🎯 Key Takeaways

1. `mod` creates a module; items are **private by default**
2. Use `pub` to make items visible outside their module
3. `use` brings items into scope to avoid long paths
4. Struct fields need individual `pub`; enum variants are all public
5. `pub use` re-exports items for cleaner APIs
6. `super::` and `self::` navigate the module tree

---
**Tomorrow:** Separating modules into files! 📂